In [0]:
# ====================================
# CELL 19: TASK 4 - PRODUCTION INFERENCE FUNCTION
# ====================================
print("Creating production inference function...")

def score_new_transactions(df_new, model_path, threshold, fraud_rate_lookup=None):
    """
    Score new transactions for fraud detection.
    
    Parameters:
    -----------
    df_new : DataFrame
        New transactions with schema matching df_log (step, type, amount, nameOrig, etc.)
    model_path : str
        Path to saved ML pipeline model
    threshold : float
        Classification threshold for fraud flag
    fraud_rate_lookup : DataFrame, optional
        Pre-computed fraud_rate by state from training data.
        If None, will use training set fraud rates.
    
    Returns:
    --------
    DataFrame with columns: nameOrig, amount, state, fraud_probability, fraud_flag
    """
    from pyspark.ml import PipelineModel
    from pyspark.ml.functions import vector_to_array
    
    print(f"Scoring {df_new.count():,} new transactions...")
    
    # Step 1: Assign state (deterministic hash)
    df_new_state = df_new.withColumn("state", assign_state_deterministic(F.col("nameOrig")))
    
    # Step 2: Enrich with government data
    df_enriched = df_new_state.join(
        F.broadcast(df_gov_agg),
        on="state",
        how="left"
    ).fillna({
        "total_pos_devices": 0,
        "total_gov_amount_lakh": 0.0
    })
    
    # Step 3: Feature engineering
    df_features = df_enriched \
        .withColumn("amount_to_gov_ratio", 
                    F.when(F.col("total_gov_amount_lakh") > 0, 
                           F.col("amount") / F.col("total_gov_amount_lakh")).otherwise(0)) \
        .withColumn("pos_density_flag", 
                    F.when(F.col("total_pos_devices") < 1000, 1).otherwise(0)) \
        .withColumn("log_gov_amount", 
                    F.when(F.col("total_gov_amount_lakh") > 0, 
                           F.log1p(F.col("total_gov_amount_lakh"))).otherwise(0)) \
        .withColumn("log_pos_devices", 
                    F.when(F.col("total_pos_devices") > 0, 
                           F.log1p(F.col("total_pos_devices"))).otherwise(0)) \
        .withColumn("balance_diff_orig", 
                    F.col("oldbalanceOrg") - F.col("newbalanceOrig")) \
        .withColumn("balance_diff_dest", 
                    F.col("newbalanceDest") - F.col("oldbalanceDest")) \
        .withColumn("orig_balance_zero_flag", 
                    F.when(F.col("oldbalanceOrg") == 0, 1).otherwise(0))
    
    # Step 4: Add fraud_rate from training (NO recomputation)
    if fraud_rate_lookup is None:
        fraud_rate_lookup = fraud_rate_train  # Use global training fraud rates
    
    df_features = df_features.join(
        F.broadcast(fraud_rate_lookup),
        on="state",
        how="left"
    ).fillna({"fraud_rate": 0.0})
    
    df_features = df_features.withColumn(
        "high_fraud_state_flag",
        F.when(F.col("fraud_rate") > 0.05, 1).otherwise(0)
    )
    
    # Step 5: Load model and predict
    print(f"Loading model from: {model_path}")
    loaded_model = PipelineModel.load(model_path)
    
    predictions = loaded_model.transform(df_features)
    
    # Step 6: Extract fraud probability and apply threshold
    predictions = predictions.withColumn(
        "fraud_probability",
        F.element_at(vector_to_array(F.col("probability")), 2)
    )
    
    output = predictions.select(
        "nameOrig",
        "amount",
        "type",
        "state",
        "fraud_probability",
        F.when(F.col("fraud_probability") >= threshold, 1).otherwise(0).alias("fraud_flag")
    )
    
    print("✓ Scoring complete")
    return output

print("✓ Function 'score_new_transactions' created")

# ====================================
# DEMO: Score a sample of test data
# ====================================
print("\n=== DEMO: Scoring sample transactions ===")

# Take a small sample from test set (simulate new data)
new_transactions_sample = test_df.limit(100).select(
    "step", "type", "amount", "nameOrig",
    "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest"
)

print(f"Sample input: {new_transactions_sample.count()} transactions")

# Score using production function
scored_transactions = score_new_transactions(
    df_new=new_transactions_sample,
    model_path=model_path,
    threshold=optimal_threshold,
    fraud_rate_lookup=fraud_rate_train
)

print("\n=== SCORED TRANSACTIONS (Top 20 by fraud probability) ===")
scored_transactions.orderBy(F.desc("fraud_probability")).show(20, truncate=False)

fraud_flagged = scored_transactions.filter(F.col("fraud_flag") == 1).count()
print(f"\n✓ Flagged {fraud_flagged} out of {scored_transactions.count()} as fraudulent")

print("\n✓ TASK 4 COMPLETE: Production inference function ready")
print("\n" + "="*60)
print("ALL TASKS COMPLETE")
print("="*60)
print("✓ Task 1: Consolidated ML Pipeline")
print("✓ Task 2: Statewise Fraud Analysis Dashboard")
print("✓ Task 3: MLflow Experiment Tracking")
print("✓ Task 4: Production Inference Function")
print("\nIssues Fixed:")
print("✓ Issue #1: Deterministic state assignment (hash-based)")
print("✓ Issue #2: Data leakage prevented (fraud_rate on train only)")
print("✓ Issue #3: Single consolidated pipeline")
print("="*60)

In [0]:
# ====================================
# CELL 18: TASK 2 - STATEWISE FRAUD ANALYSIS DASHBOARD
# ====================================
print("Building statewise fraud analysis summary table...")

# Aggregate predictions by state (using fraud_probability column from predictions_final)
state_summary = predictions_final.groupBy("state").agg(
    F.count("*").alias("total_transactions"),
    F.sum("is_fraud").alias("fraud_count"),
    (F.sum("is_fraud") / F.count("*") * 100).alias("fraud_rate_pct"),
    F.mean("amount").alias("avg_transaction_amount"),
    F.mean("fraud_probability").alias("avg_prob_fraud"),
    F.first("total_gov_amount_lakh").alias("total_gov_amount_lakh"),
    F.first("total_pos_devices").alias("total_pos_devices")
)

# Add risk tier classification
state_summary = state_summary.withColumn(
    "risk_tier",
    F.when(F.col("fraud_rate_pct") > 10, "HIGH")
     .when(F.col("fraud_rate_pct") >= 5, "MEDIUM")
     .otherwise("LOW")
)

# Order by fraud rate (descending)
state_summary = state_summary.orderBy(F.desc("fraud_rate_pct"))

print("\n=== STATEWISE FRAUD ANALYSIS SUMMARY ===")
state_summary.show(20, truncate=False)

# Risk tier distribution
print("\n=== RISK TIER DISTRIBUTION ===")
risk_dist = state_summary.groupBy("risk_tier").agg(
    F.count("*").alias("num_states"),
    F.sum("total_transactions").alias("total_txns"),
    F.sum("fraud_count").alias("total_fraud")
).orderBy(F.desc("num_states"))
risk_dist.show()

# Save to Delta table for dashboard consumption
print("\nSaving to Unity Catalog...")
state_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.upi_fraud_state_summary")

print("✓ Dashboard table created: workspace.default.upi_fraud_state_summary")
print("\nTable Schema:")
print("  - state: State/UT name")
print("  - total_transactions: Total transaction count")
print("  - fraud_count: Number of fraudulent transactions")
print("  - fraud_rate_pct: Fraud rate percentage")
print("  - avg_transaction_amount: Average transaction amount")
print("  - total_gov_amount_lakh: Total government sanctioned amount (lakh)")
print("  - total_pos_devices: Total PoS devices sanctioned")
print("  - avg_prob_fraud: Average fraud probability from model")
print("  - risk_tier: HIGH / MEDIUM / LOW")
print("\n✓ TASK 2 COMPLETE: Dashboard summary table ready for SQL queries")

In [0]:
# ====================================
# CELL 14: THRESHOLD TUNING (3-TIER FALLBACK)
# ====================================
print("Tuning classification threshold...")
print("\nThreshold tuning with 3-tier fallback logic:")
print("  Tier 1: Maximize F1 score")
print("  Tier 2: If F1 tie, maximize recall")
print("  Tier 3: If recall tie, minimize false positives\n")

thresholds = [i/10 for i in range(2, 10)]  # 0.2 to 0.9
threshold_results = []

for threshold in thresholds:
    # Apply threshold (using fraud_probability column from Cell 12)
    preds = predictions_test.withColumn(
        "pred_adj",
        F.when(F.col("fraud_probability") >= threshold, 1).otherwise(0)
    )
    
    # Calculate metrics
    tp_count = preds.filter((F.col("is_fraud") == 1) & (F.col("pred_adj") == 1)).count()
    fp_count = preds.filter((F.col("is_fraud") == 0) & (F.col("pred_adj") == 1)).count()
    fn_count = preds.filter((F.col("is_fraud") == 1) & (F.col("pred_adj") == 0)).count()
    tn_count = preds.filter((F.col("is_fraud") == 0) & (F.col("pred_adj") == 0)).count()
    
    precision_t = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0
    recall_t = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0
    f1_t = 2 * (precision_t * recall_t) / (precision_t + recall_t) if (precision_t + recall_t) > 0 else 0
    
    threshold_results.append({
        "threshold": threshold,
        "tp": tp_count,
        "fp": fp_count,
        "fn": fn_count,
        "tn": tn_count,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t
    })

# Create DataFrame
threshold_df = spark.createDataFrame(threshold_results)
print("=== THRESHOLD TUNING RESULTS ===")
threshold_df.orderBy(F.desc("f1")).show(truncate=False)

# 3-tier fallback selection
# Tier 1: Max F1
max_f1 = threshold_df.agg(F.max("f1")).collect()[0][0]
candidates = threshold_df.filter(F.col("f1") == max_f1)

if candidates.count() == 1:
    optimal = candidates.first()
    print(f"\n✓ Tier 1: Unique F1 maximum at threshold={optimal['threshold']}")
else:
    # Tier 2: Max recall among F1 ties
    max_recall = candidates.agg(F.max("recall")).collect()[0][0]
    candidates = candidates.filter(F.col("recall") == max_recall)
    
    if candidates.count() == 1:
        optimal = candidates.first()
        print(f"\n✓ Tier 2: Max recall among F1 ties at threshold={optimal['threshold']}")
    else:
        # Tier 3: Min FP among recall ties
        min_fp = candidates.agg(F.min("fp")).collect()[0][0]
        optimal = candidates.filter(F.col("fp") == min_fp).first()
        print(f"\n✓ Tier 3: Min FP among recall ties at threshold={optimal['threshold']}")

optimal_threshold = optimal["threshold"]
print(f"\n=== OPTIMAL THRESHOLD: {optimal_threshold} ===")
print(f"F1: {optimal['f1']:.4f}")
print(f"Precision: {optimal['precision']:.4f}")
print(f"Recall: {optimal['recall']:.4f}")
print(f"TP: {optimal['tp']}, FP: {optimal['fp']}, FN: {optimal['fn']}, TN: {optimal['tn']}")

# Log to MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_param("optimal_threshold", optimal_threshold)
    mlflow.log_metric("optimal_f1", optimal['f1'])
    mlflow.log_metric("optimal_precision", optimal['precision'])
    mlflow.log_metric("optimal_recall", optimal['recall'])

print("\n✓ Optimal threshold logged to MLflow")

Tuning classification threshold...

Threshold tuning with 3-tier fallback logic:
  Tier 1: Maximize F1 score
  Tier 2: If F1 tie, maximize recall
  Tier 3: If recall tie, minimize false positives

=== THRESHOLD TUNING RESULTS ===
+--------------------+---+------+--------------------+------------------+---------+-------+----+
|f1                  |fn |fp    |precision           |recall            |threshold|tn     |tp  |
+--------------------+---+------+--------------------+------------------+---------+-------+----+
|0.21833583822926628 |28 |11414 |0.12280971411005226 |0.982779827798278 |0.9      |1259167|1598|
|0.14087666561981246 |13 |19136 |0.07582343282140443 |0.9917877447883765|0.8      |1251599|1570|
|0.10033015407190021 |10 |29420 |0.05283152506358456 |0.9939430648092066|0.7      |1240849|1641|
|0.07679481682625885 |5  |38753 |0.03993558776167472 |0.9969078540507111|0.6      |1231738|1612|
|0.0556748952354122  |7  |53625 |0.028638191500923813|0.9955919395465995|0.5      |1217425|

In [0]:
# ====================================
# CELL 15: CROSS-VALIDATION
# ====================================
print("Running cross-validation with parameter grid...")

# Create new pipeline for CV (using balanced training set)
rf_cv = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_fraud",
    weightCol="sample_weight",
    seed=42,
    rawPredictionCol="rawPrediction",
    probabilityCol="probability"
)

# Build pipeline for CV
stages_cv = stages[:-1]  # All stages except the final RF
stages_cv.append(rf_cv)
pipeline_cv = Pipeline(stages=stages_cv)

# Parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(rf_cv.numTrees, [50, 100]) \
    .addGrid(rf_cv.maxDepth, [5, 10]) \
    .build()

print(f"Parameter grid size: {len(paramGrid)}")

# Evaluator
evaluator_cv = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Cross-validator (3-fold)
cv = CrossValidator(
    estimator=pipeline_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_cv,
    numFolds=3,
    seed=42
)

print("\nRunning 3-fold cross-validation...")
print("This may take several minutes...")

# Memory cleanup
gc.collect()

# Fit CV
cv_model = cv.fit(train_balanced)

print("✓ Cross-validation complete")

# Best parameters
best_model_cv = cv_model.bestModel
best_rf = best_model_cv.stages[-1]

print(f"\n=== BEST MODEL PARAMETERS ===")
print(f"numTrees: {best_rf.getNumTrees}")
print(f"maxDepth: {best_rf.getMaxDepth()}")

# CV metrics
avg_metrics = cv_model.avgMetrics
print(f"\nCross-validation AUC-ROC scores: {[f'{m:.4f}' for m in avg_metrics]}")
print(f"Best CV AUC-ROC: {max(avg_metrics):.4f}")

# Log to MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_param("cv_best_num_trees", best_rf.getNumTrees)
    mlflow.log_param("cv_best_max_depth", best_rf.getMaxDepth())
    mlflow.log_metric("cv_best_auc", max(avg_metrics))

print("✓ CV results logged to MLflow")

In [0]:
# ====================================
# CELL 16: SAVE MODEL AND RESULTS
# ====================================
print("Saving model and results...")

# Save to Unity Catalog with MLflow (primary storage)
# For serverless: requires UC Volume path for temporary storage
with mlflow.start_run(run_id=run_id):
    # Create input example for UC signature (required)
    input_sample = train_balanced.limit(5)
    input_example = input_sample.toPandas()
    
    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        registered_model_name="workspace.default.upi_fraud_detector",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp/",
        input_example=input_example
    )

print("✓ Model registered in Unity Catalog: workspace.default.upi_fraud_detector")
print("   Use models:/workspace.default.upi_fraud_detector/latest for production inference")

# For model_path variable (used in Cell 19), use Unity Catalog reference
model_path = "models:/workspace.default.upi_fraud_detector/latest"

# Save enriched final data to Delta
print("\nSaving enriched dataset to Delta table...")
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.upi_fraud_enriched_final")

print("✓ Data saved to: workspace.default.upi_fraud_enriched_final")

# Save threshold results
print("\nSaving threshold tuning results...")
threshold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.upi_fraud_threshold_results")

print("✓ Threshold results saved to: workspace.default.upi_fraud_threshold_results")

print("\n" + "="*60)
print("MODEL TRAINING COMPLETE")
print("="*60)
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Optimal F1: {optimal['f1']:.4f}")
print(f"Optimal Threshold: {optimal_threshold}")
print(f"\nProduction Criteria Check:")
print(f"  Precision ≥ 0.80: {'FAIL' if optimal['precision'] < 0.80 else 'PASS'} ({optimal['precision']:.4f})")
print(f"  Recall ≥ 0.70: {'PASS' if optimal['recall'] >= 0.70 else 'FAIL'} ({optimal['recall']:.4f})")
print(f"\nNote: Low precision (12.3%) indicates many false positives.")
print(f"      High recall (98.3%) means we catch almost all fraud.")
print(f"      This trade-off prioritizes fraud detection over false alarms.")

Saving model and results...


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/18 06:49:05 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==

Uploading artifacts:   0%|          | 0/46 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.upi_fraud_detector': https://dbc-db8911d3-57ab.cloud.databricks.com/explore/data/models/workspace/default/upi_fraud_detector/version/1?o=7474659941313398


✓ Model registered in Unity Catalog: workspace.default.upi_fraud_detector
   Use models:/workspace.default.upi_fraud_detector/latest for production inference

Saving enriched dataset to Delta table...
✓ Data saved to: workspace.default.upi_fraud_enriched_final

Saving threshold tuning results...
✓ Threshold results saved to: workspace.default.upi_fraud_threshold_results

MODEL TRAINING COMPLETE
AUC-ROC: 0.9986
PR-AUC: 0.8144
Optimal F1: 0.2183
Optimal Threshold: 0.9

Production Criteria Check:
  Precision ≥ 0.80: FAIL (0.1228)
  Recall ≥ 0.70: PASS (0.9828)

Note: Low precision (12.3%) indicates many false positives.
      High recall (98.3%) means we catch almost all fraud.
      This trade-off prioritizes fraud detection over false alarms.


In [0]:
# ====================================
# CELL 17: INFERENCE DEMO (TASK 1 COMPLETE)
# ====================================
print("Demonstrating inference on test data...")

# Apply optimal threshold (using fraud_probability column)
predictions_final = predictions_test.withColumn(
    "fraud_flag",
    F.when(F.col("fraud_probability") >= optimal_threshold, 1).otherwise(0)
)

# Select key columns for output
inference_output = predictions_final.select(
    "nameOrig",
    "amount",
    "type",
    "state",
    "is_fraud",
    "fraud_probability",
    "fraud_flag"
).orderBy(F.desc("fraud_probability"))

print("\n=== TOP 20 HIGH-RISK TRANSACTIONS ===")
inference_output.show(20, truncate=False)

# Statistics
total_flagged = inference_output.filter(F.col("fraud_flag") == 1).count()
total_test = inference_output.count()
flag_rate = (total_flagged / total_test) * 100

print(f"\nInference Statistics:")
print(f"  Total test transactions: {total_test:,}")
print(f"  Flagged as fraud: {total_flagged:,} ({flag_rate:.2f}%)")

# Actual fraud in flagged transactions
tp_flagged = inference_output.filter((F.col("fraud_flag") == 1) & (F.col("is_fraud") == 1)).count()
if total_flagged > 0:
    precision_flagged = (tp_flagged / total_flagged) * 100
    print(f"  Actual fraud in flagged: {tp_flagged:,} ({precision_flagged:.1f}% precision)")

print("\n✓ Inference demo complete")
print("✓ TASK 1 COMPLETE: Consolidated ML Pipeline Ready")

Demonstrating inference on test data...

=== TOP 20 HIGH-RISK TRANSACTIONS ===
+-----------+----------+--------+-----------------+--------+------------------+----------+
|nameOrig   |amount    |type    |state            |is_fraud|fraud_probability |fraud_flag|
+-----------+----------+--------+-----------------+--------+------------------+----------+
|C503443667 |3242183.64|TRANSFER|Bihar            |1       |0.99999300521924  |1         |
|C676092302 |2406332.79|TRANSFER|Punjab Total     |1       |0.9999929914394984|1         |
|C1952841743|1058602.68|TRANSFER|Himachal Pradesh |1       |0.9999929914394984|1         |
|C529879129 |5455965.86|TRANSFER|Jammu and Kashmir|1       |0.9999929668712144|1         |
|C571696283 |4017972.88|TRANSFER|Bihar            |1       |0.9999929540657004|1         |
|C829335128 |2763025.35|TRANSFER|Bihar            |1       |0.9999929540657004|1         |
|C1908904298|7418265.31|TRANSFER|Punjab Total     |1       |0.9999929402859586|1         |
|C574846060

In [0]:
# ====================================
# CELL 10: DEFINE ML PIPELINE
# ====================================
print("Building ML Pipeline...")

# Define feature columns
feature_cols = [
    "step", "amount",
    "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "total_pos_devices", "total_gov_amount_lakh",
    "fraud_rate", "amount_to_gov_ratio",
    "pos_density_flag", "high_fraud_state_flag",
    "log_gov_amount", "log_pos_devices",
    "balance_diff_orig", "balance_diff_dest",
    "orig_balance_zero_flag"
]

categorical_cols = ["type"]

print(f"Numeric features ({len(feature_cols)}): {feature_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Pipeline stages
stages = []

# Stage 1: StringIndexer for categorical features
for cat_col in categorical_cols:
    indexer = StringIndexer(
        inputCol=cat_col,
        outputCol=f"{cat_col}_index",
        handleInvalid="keep"
    )
    stages.append(indexer)
    feature_cols.append(f"{cat_col}_index")

print(f"\n✓ StringIndexer stages: {len(categorical_cols)}")

# Stage 2: VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_unscaled",
    handleInvalid="skip"
)
stages.append(assembler)
print("✓ VectorAssembler added")

# Stage 3: StandardScaler
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=False
)
stages.append(scaler)
print("✓ StandardScaler added")

# Stage 4: RandomForestClassifier
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_fraud",
    weightCol="sample_weight",
    numTrees=100,
    maxDepth=10,
    seed=42,
    rawPredictionCol="rawPrediction",
    probabilityCol="probability"
)
stages.append(rf)
print("✓ RandomForestClassifier added")

# Create pipeline
pipeline = Pipeline(stages=stages)

print(f"\n✓ Pipeline created with {len(stages)} stages")
print("   1. StringIndexer (type)")
print("   2. VectorAssembler")
print("   3. StandardScaler")
print("   4. RandomForestClassifier")

Building ML Pipeline...
Numeric features (17): ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'total_pos_devices', 'total_gov_amount_lakh', 'fraud_rate', 'amount_to_gov_ratio', 'pos_density_flag', 'high_fraud_state_flag', 'log_gov_amount', 'log_pos_devices', 'balance_diff_orig', 'balance_diff_dest', 'orig_balance_zero_flag']
Categorical features (1): ['type']

✓ StringIndexer stages: 1
✓ VectorAssembler added
✓ StandardScaler added
✓ RandomForestClassifier added

✓ Pipeline created with 4 stages
   1. StringIndexer (type)
   2. VectorAssembler
   3. StandardScaler
   4. RandomForestClassifier


In [0]:
# ====================================
# CELL 11: TRAIN MODEL (TASK 3: MLflow)
# ====================================
print("Training model with MLflow tracking...")

# Set MLflow experiment
mlflow.set_experiment("/Users/sse240021019@iiti.ac.in/upi_fraud_detection")

with mlflow.start_run(run_name="upi_fraud_rf_v1") as run:
    # Log parameters
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("downsample_fraction", downsample_fraction)
    mlflow.log_param("target_ratio", target_ratio)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("model_type", "RandomForestClassifier")
    
    # Memory cleanup before training
    gc.collect()
    
    # Train model
    print("\nTraining pipeline...")
    model = pipeline.fit(train_balanced)
    
    print("✓ Model training complete")
    
    # Save run_id for later use
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

print("\n✓ Model ready for evaluation")

Training model with MLflow tracking...


2026/04/18 06:16:55 INFO mlflow.tracking.fluent: Experiment with name '/Users/sse240021019@iiti.ac.in/upi_fraud_detection' does not exist. Creating a new experiment.



Training pipeline...
✓ Model training complete
MLflow Run ID: b2c49b8e349b4f7a8021520195902575

✓ Model ready for evaluation


In [0]:
# ====================================
# CELL 12: MODEL EVALUATION
# ====================================
print("Evaluating model on test set...")

# Make predictions on test set
predictions_test = model.transform(test_df)

print("✓ Predictions generated")

# Evaluators
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

# AUC-ROC and PR-AUC
auc_roc = auc_evaluator.evaluate(predictions_test)
pr_auc = pr_evaluator.evaluate(predictions_test)

print(f"\n=== MODEL PERFORMANCE ===")
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

# Extract fraud probability using vector_to_array and element_at
from pyspark.ml.functions import vector_to_array

predictions_test = predictions_test.withColumn(
    "fraud_probability",
    F.element_at(vector_to_array(F.col("probability")), 2)
)

# Add prediction column (threshold = 0.5 default) - cast to double for evaluator
predictions_test = predictions_test.withColumn(
    "prediction",
    F.when(F.col("fraud_probability") >= 0.5, 1.0).otherwise(0.0)
)

# Multiclass metrics
mc_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_fraud",
    predictionCol="prediction"
)

accuracy = mc_evaluator.evaluate(predictions_test, {mc_evaluator.metricName: "accuracy"})
f1 = mc_evaluator.evaluate(predictions_test, {mc_evaluator.metricName: "f1"})
precision = mc_evaluator.evaluate(predictions_test, {mc_evaluator.metricName: "weightedPrecision"})
recall = mc_evaluator.evaluate(predictions_test, {mc_evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

# Confusion Matrix
print("\n=== CONFUSION MATRIX ===")
confusion = predictions_test.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction")
confusion.show()

# Manual confusion matrix breakdown
tn = predictions_test.filter((F.col("is_fraud") == 0) & (F.col("prediction") == 0)).count()
fp = predictions_test.filter((F.col("is_fraud") == 0) & (F.col("prediction") == 1)).count()
fn = predictions_test.filter((F.col("is_fraud") == 1) & (F.col("prediction") == 0)).count()
tp = predictions_test.filter((F.col("is_fraud") == 1) & (F.col("prediction") == 1)).count()

print(f"True Negatives: {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives: {tp:,}")

if (tp + fp) > 0:
    precision_manual = tp / (tp + fp)
    print(f"\nManual Precision: {precision_manual:.4f}")

if (tp + fn) > 0:
    recall_manual = tp / (tp + fn)
    print(f"Manual Recall: {recall_manual:.4f}")

# Log metrics to MLflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("auc_roc", auc_roc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("true_positives", tp)
    mlflow.log_metric("false_positives", fp)
    mlflow.log_metric("false_negatives", fn)
    mlflow.log_metric("true_negatives", tn)

print("\n✓ Metrics logged to MLflow")

Evaluating model on test set...
✓ Predictions generated

=== MODEL PERFORMANCE ===
AUC-ROC: 0.9986
PR-AUC: 0.8144
Accuracy: 0.9584
F1 Score: 0.9773
Precision: 0.9988
Recall: 0.9588

=== CONFUSION MATRIX ===
+--------+----------+-------+
|is_fraud|prediction|  count|
+--------+----------+-------+
|       0|       0.0|1217146|
|       0|       1.0|  53453|
|       1|       0.0|      4|
|       1|       1.0|   1622|
+--------+----------+-------+

True Negatives: 1,216,850
False Positives: 54,252
False Negatives: 4
True Positives: 1,594

Manual Precision: 0.0285
Manual Recall: 0.9975

✓ Metrics logged to MLflow


In [0]:
# ====================================
# CELL 13: FEATURE IMPORTANCE
# ====================================
print("Extracting feature importance...")

# Get RandomForest model from pipeline
rf_model = model.stages[-1]

# Get feature importances
importances = rf_model.featureImportances.toArray()

# Create DataFrame with explicit type conversion
importance_data = [(str(feat), float(imp)) for feat, imp in zip(feature_cols, importances)]
importance_df = spark.createDataFrame(importance_data, ["feature", "importance"]) \
    .orderBy(F.desc("importance"))

print("\n=== TOP 15 FEATURES ===")
importance_df.show(15, truncate=False)

# Convert to pandas for MLflow artifact
importance_pd = importance_df.toPandas()

# Save to CSV and log to MLflow
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = os.path.join(tmpdir, "feature_importance.csv")
    importance_pd.to_csv(csv_path, index=False)
    
    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifact(csv_path)

print("✓ Feature importance logged to MLflow")

# Summary statistics
top_5_importance = importance_pd.head(5)["importance"].sum()
print(f"\nTop 5 features account for {top_5_importance*100:.1f}% of total importance")

Extracting feature importance...

=== TOP 15 FEATURES ===
+----------------------+--------------------+
|feature               |importance          |
+----------------------+--------------------+
|amount                |0.18929782865760666 |
|type_index            |0.1457881974609307  |
|newbalanceOrig        |0.10454746448553745 |
|balance_diff_orig     |0.0939380318971262  |
|balance_diff_dest     |0.08373371424345245 |
|oldbalanceOrg         |0.07864714604858952 |
|step                  |0.0657178738330613  |
|orig_balance_zero_flag|0.057026955987337924|
|oldbalanceDest        |0.04994700285808538 |
|newbalanceDest        |0.03948817694006522 |
|amount_to_gov_ratio   |0.03550664943439562 |
|fraud_rate            |0.022154397478051018|
|log_gov_amount        |0.010164743858909026|
|log_pos_devices       |0.0090932707809078  |
|total_gov_amount_lakh |0.008452461161959449|
+----------------------+--------------------+
only showing top 15 rows
✓ Feature importance logged to MLflow

Top 

In [0]:
# ====================================
# CELL 6: FEATURE ENGINEERING (NO LEAKAGE)
# ====================================
print("Creating derived features...")

# Basic derived features (no data leakage)
df_features = df_enriched \
    .withColumn("amount_to_gov_ratio", 
                F.when(F.col("total_gov_amount_lakh") > 0, 
                       F.col("amount") / F.col("total_gov_amount_lakh")).otherwise(0)) \
    .withColumn("pos_density_flag", 
                F.when(F.col("total_pos_devices") < 1000, 1).otherwise(0)) \
    .withColumn("log_gov_amount", 
                F.when(F.col("total_gov_amount_lakh") > 0, 
                       F.log1p(F.col("total_gov_amount_lakh"))).otherwise(0)) \
    .withColumn("log_pos_devices", 
                F.when(F.col("total_pos_devices") > 0, 
                       F.log1p(F.col("total_pos_devices"))).otherwise(0)) \
    .withColumn("balance_diff_orig", 
                F.col("oldbalanceOrg") - F.col("newbalanceOrig")) \
    .withColumn("balance_diff_dest", 
                F.col("newbalanceDest") - F.col("oldbalanceDest")) \
    .withColumn("orig_balance_zero_flag", 
                F.when(F.col("oldbalanceOrg") == 0, 1).otherwise(0))

# Fill nulls in government data columns with 0
df_features = df_features.fillna({
    "total_pos_devices": 0,
    "total_gov_amount_lakh": 0.0
})

# Remove duplicates
df_features = df_features.dropDuplicates()

print(f"✓ Feature engineering complete: {df_features.count():,} rows")
print("\nFeature columns created:")
print("  - amount_to_gov_ratio, pos_density_flag, log_gov_amount, log_pos_devices")
print("  - balance_diff_orig, balance_diff_dest, orig_balance_zero_flag")
print("\n⚠️ fraud_rate will be computed AFTER train/test split (Cell 8) to prevent leakage.")

Creating derived features...
✓ Feature engineering complete: 6,362,620 rows

Feature columns created:
  - amount_to_gov_ratio, pos_density_flag, log_gov_amount, log_pos_devices
  - balance_diff_orig, balance_diff_dest, orig_balance_zero_flag

⚠️ fraud_rate will be computed AFTER train/test split (Cell 8) to prevent leakage.


In [0]:
# ====================================
# CELL 7: EXPLORATORY DATA ANALYSIS
# ====================================
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Class balance
print("\n1. CLASS BALANCE")
class_dist = df_features.groupBy("is_fraud").count().orderBy("is_fraud")
class_dist.show()

total = df_features.count()
fraud_count = df_features.filter(F.col("is_fraud") == 1).count()
non_fraud_count = total - fraud_count
fraud_pct = (fraud_count / total) * 100

print(f"Total transactions: {total:,}")
print(f"Fraud: {fraud_count:,} ({fraud_pct:.2f}%)")
print(f"Non-fraud: {non_fraud_count:,} ({100-fraud_pct:.2f}%)")
print(f"Imbalance ratio: 1:{non_fraud_count/fraud_count:.1f}")

# Fraud by state
print("\n2. FRAUD BY STATE (Top 10)")
fraud_by_state = df_features.groupBy("state").agg(
    F.count("*").alias("total_txns"),
    F.sum("is_fraud").alias("fraud_count"),
    (F.sum("is_fraud") / F.count("*") * 100).alias("fraud_rate_pct")
).orderBy(F.desc("fraud_rate_pct"))
fraud_by_state.show(10, truncate=False)

# Fraud by transaction type
print("\n3. FRAUD BY TRANSACTION TYPE")
fraud_by_type = df_features.groupBy("type").agg(
    F.count("*").alias("total_txns"),
    F.sum("is_fraud").alias("fraud_count"),
    (F.sum("is_fraud") / F.count("*") * 100).alias("fraud_rate_pct")
).orderBy(F.desc("fraud_rate_pct"))
fraud_by_type.show()

# Amount statistics
print("\n4. TRANSACTION AMOUNT ANALYSIS")
amount_stats = df_features.groupBy("is_fraud").agg(
    F.mean("amount").alias("avg_amount"),
    F.stddev("amount").alias("std_amount"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount")
)
amount_stats.show(truncate=False)

print("✓ EDA complete")

EXPLORATORY DATA ANALYSIS

1. CLASS BALANCE
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       0|6354407|
|       1|   8213|
+--------+-------+

Total transactions: 6,362,620
Fraud: 8,213 (0.13%)
Non-fraud: 6,354,407 (99.87%)
Imbalance ratio: 1:773.7

2. FRAUD BY STATE (Top 10)
+---------------------------+----------+-----------+-------------------+
|state                      |total_txns|fraud_count|fraud_rate_pct     |
+---------------------------+----------+-----------+-------------------+
|Andaman and Nicobar Islands|171710    |251        |0.14617669326189506|
|Andhra Pradesh             |171920    |251        |0.14599813866914843|
|Odisha                     |172040    |248        |0.14415252266914672|
|Chandigarh                 |171910    |247        |0.14367983247047875|
|Sikkim                     |172069    |243        |0.14122241658869408|
|Maharashtra                |172376    |242        |0.14039077365758573|
|West Bengal                |171859    |239       

In [0]:
# ====================================
# CELL 8: TRAIN/TEST SPLIT + FRAUD_RATE (FIX ISSUE #2)
# ====================================
print("Performing train/test split BEFORE computing fraud_rate...")

# Split data 80/20
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

print(f"✓ Train set: {train_df.count():,} rows")
print(f"✓ Test set: {test_df.count():,} rows")

# Compute fraud_rate ONLY on training data (prevents data leakage)
print("\nComputing fraud_rate on training set only...")
fraud_rate_train = train_df.groupBy("state").agg(
    F.mean("is_fraud").alias("fraud_rate")
)

print(f"✓ Fraud rates computed for {fraud_rate_train.count()} states")

# Join fraud_rate back to train and test sets
train_df = train_df.drop("fraud_rate").join(
    F.broadcast(fraud_rate_train),
    on="state",
    how="left"
).fillna({"fraud_rate": 0.0})

test_df = test_df.drop("fraud_rate").join(
    F.broadcast(fraud_rate_train),  # Use TRAIN fraud_rates for test set too
    on="state",
    how="left"
).fillna({"fraud_rate": 0.0})

# Add high_fraud_state_flag based on fraud_rate
train_df = train_df.withColumn("high_fraud_state_flag", 
                                F.when(F.col("fraud_rate") > 0.05, 1).otherwise(0))
test_df = test_df.withColumn("high_fraud_state_flag", 
                              F.when(F.col("fraud_rate") > 0.05, 1).otherwise(0))

print("✓ fraud_rate and high_fraud_state_flag added without data leakage")
print("\nFraud rate distribution in training set:")
train_df.select("fraud_rate").describe().show()

Performing train/test split BEFORE computing fraud_rate...
✓ Train set: 5,090,379 rows
✓ Test set: 1,272,229 rows

Computing fraud_rate on training set only...
✓ Fraud rates computed for 37 states
✓ fraud_rate and high_fraud_state_flag added without data leakage

Fraud rate distribution in training set:
+-------+--------------------+
|summary|          fraud_rate|
+-------+--------------------+
|  count|             5090425|
|   mean|0.001285157919037805|
| stddev|9.374995471407104E-5|
|    min|0.001075597029026...|
|    max|0.001472872173653...|
+-------+--------------------+



In [0]:
# ====================================
# CELL 9: HANDLE CLASS IMBALANCE
# ====================================
print("Handling class imbalance with auto-computed downsampling...")

# Count fraud and non-fraud in training set
fraud_train = train_df.filter(F.col("is_fraud") == 1).count()
non_fraud_train = train_df.filter(F.col("is_fraud") == 0).count()

print(f"Training set class distribution:")
print(f"  - Fraud: {fraud_train:,}")
print(f"  - Non-fraud: {non_fraud_train:,}")
print(f"  - Imbalance ratio: 1:{non_fraud_train/fraud_train:.1f}")

# Auto-compute downsampling fraction (target ratio = 5:1)
target_ratio = 5.0
target_non_fraud = int(fraud_train * target_ratio)
downsample_fraction = target_non_fraud / non_fraud_train

print(f"\nTarget ratio: 1:{target_ratio}")
print(f"Target non-fraud count: {target_non_fraud:,}")
print(f"Downsampling fraction: {downsample_fraction:.4f}")

# Apply downsampling to non-fraud class
fraud_df = train_df.filter(F.col("is_fraud") == 1)
non_fraud_df = train_df.filter(F.col("is_fraud") == 0).sample(
    withReplacement=False,
    fraction=downsample_fraction,
    seed=42
)

# Combine
train_balanced = fraud_df.union(non_fraud_df)

print(f"\n✓ Balanced training set: {train_balanced.count():,} rows")
balanced_dist = train_balanced.groupBy("is_fraud").count().orderBy("is_fraud")
balanced_dist.show()

# Compute class weights for sample weighting
class_weight_0 = 1.0
class_weight_1 = non_fraud_train / fraud_train

train_balanced = train_balanced.withColumn(
    "sample_weight",
    F.when(F.col("is_fraud") == 1, class_weight_1).otherwise(class_weight_0)
)

print(f"✓ Sample weights added: fraud={class_weight_1:.2f}, non-fraud={class_weight_0:.2f}")
print(f"\nMemory cleanup...")
gc.collect()
print("✓ Ready for model training")

Handling class imbalance with auto-computed downsampling...
Training set class distribution:
  - Fraud: 6,543
  - Non-fraud: 5,083,799
  - Imbalance ratio: 1:777.0

Target ratio: 1:5.0
Target non-fraud count: 32,715
Downsampling fraction: 0.0064

✓ Balanced training set: 39,038 rows
+--------+-----+
|is_fraud|count|
+--------+-----+
|       0|32439|
|       1| 6535|
+--------+-----+

✓ Sample weights added: fraud=776.98, non-fraud=1.00

Memory cleanup...
✓ Ready for model training


In [0]:
# ====================================
# CELL 1: IMPORTS AND SETUP
# ====================================
import gc
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import mlflow
import mlflow.spark

print("✓ All imports loaded successfully")
print(f"MLflow version: {mlflow.__version__}")
print(f"Spark version: {spark.version}")

✓ All imports loaded successfully
MLflow version: 3.8.1
Spark version: 4.1.0


In [0]:
# ====================================
# CELL 2: LOAD SOURCE TABLES
# ====================================
print("Loading source tables from Unity Catalog...")

# Load transaction log (Kaggle dataset)
df_log = spark.read.table("workspace.default.ps_20174392719_1491204439457_log")
print(f"✓ Transaction log loaded: {df_log.count():,} rows")

# Load government infrastructure data
df_gov = spark.read.table("workspace.default.session_244_as_162_1_1")
print(f"✓ Government data loaded: {df_gov.count():,} rows")

print("\n=== Transaction Log Schema ===")
df_log.printSchema()
print("\n=== Government Data Schema ===")
df_gov.printSchema()

Loading source tables from Unity Catalog...
✓ Transaction log loaded: 6,362,620 rows
✓ Government data loaded: 37 rows

=== Transaction Log Schema ===
root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)


=== Government Data Schema ===
root
 |-- Sr.No: string (nullable = true)
 |-- State/UT: string (nullable = true)
 |-- No. of PoS devices sanctioned: long (nullable = true)
 |-- Amount sanctioned in lakh: double (nullable = true)



In [0]:
# ====================================
# CELL 3: CLEAN AND AGGREGATE GOV DATA
# ====================================
print("Cleaning government infrastructure data...")

# Rename columns for consistency
df_gov_clean = df_gov \
    .withColumnRenamed("State/UT", "state") \
    .withColumnRenamed("No. of PoS devices sanctioned", "pos_devices") \
    .withColumnRenamed("Amount sanctioned in lakh", "gov_amount")

# Cast to proper types
df_gov_clean = df_gov_clean \
    .withColumn("pos_devices", F.col("pos_devices").cast(IntegerType())) \
    .withColumn("gov_amount", F.col("gov_amount").cast(DoubleType())) \
    .filter(F.col("state").isNotNull())

# Aggregate by state (sum of devices and amount)
df_gov_agg = df_gov_clean.groupBy("state").agg(
    F.sum("pos_devices").alias("total_pos_devices"),
    F.sum("gov_amount").alias("total_gov_amount_lakh")
)

print(f"✓ Government data aggregated: {df_gov_agg.count()} unique states")
print("\nSample aggregated government data:")
df_gov_agg.show(5, truncate=False)

Cleaning government infrastructure data...
✓ Government data aggregated: 37 unique states

Sample aggregated government data:
+---------------------------+-----------------+---------------------+
|state                      |total_pos_devices|total_gov_amount_lakh|
+---------------------------+-----------------+---------------------+
|Andaman and Nicobar Islands|194              |11.64                |
|Andhra Pradesh             |7594             |455.64               |
|Arunachal Pradesh          |161              |9.66                 |
|Assam                      |4336             |260.16               |
|Bihar                      |11498            |689.88               |
+---------------------------+-----------------+---------------------+
only showing top 5 rows


In [0]:
# ====================================
# CELL 4: FIX STATE ASSIGNMENT - ISSUE #1
# ====================================
# PROBLEM: Original dataset has NO state information.
# SOLUTION: Hash-based deterministic state assignment using nameOrig.
# This ensures reproducibility across runs (same customer → same state).

print("Applying deterministic hash-based state assignment...")

# Get list of states from government data
states_list = [row.state for row in df_gov_agg.select("state").distinct().collect()]
print(f"Available states: {len(states_list)}")

# Create UDF for deterministic state assignment
# Using hash of nameOrig modulo number of states
@F.udf(returnType=StringType())
def assign_state_deterministic(name_orig):
    if name_orig is None:
        return None
    hash_val = hash(name_orig)
    return states_list[hash_val % len(states_list)]

# Apply state assignment
df_log_with_state = df_log.withColumn("state", assign_state_deterministic(F.col("nameOrig")))

# Rename fraud columns for consistency
df_log_with_state = df_log_with_state \
    .withColumnRenamed("isFraud", "is_fraud") \
    .withColumnRenamed("isFlaggedFraud", "is_flagged_fraud")

print(f"✓ State assignment complete")
print("\nState distribution:")
df_log_with_state.groupBy("state").count().orderBy(F.desc("count")).show(10)

print("\n⚠️ NOTE: State assignment is synthetic (hash-based) since source data lacks geography.")
print("   fraud_rate and state-based features reflect this synthetic mapping.")

Applying deterministic hash-based state assignment...
Available states: 37
✓ State assignment complete

State distribution:
+-----------------+------+
|            state| count|
+-----------------+------+
|Arunachal Pradesh|172772|
|          Mizoram|172702|
|           Kerala|172667|
|     Punjab Total|172487|
|          Tripura|172482|
|          Manipur|172411|
| Himachal Pradesh|172396|
|      Maharashtra|172240|
|Jammu and Kashmir|172210|
|      Grand Total|172168|
+-----------------+------+
only showing top 10 rows

⚠️ NOTE: State assignment is synthetic (hash-based) since source data lacks geography.
   fraud_rate and state-based features reflect this synthetic mapping.


In [0]:
# ====================================
# CELL 5: LEFT JOIN WITH GOV DATA
# ====================================
print("Joining transaction log with government infrastructure data...")

# Use broadcast join (df_gov_agg is small)
df_enriched = df_log_with_state.join(
    F.broadcast(df_gov_agg),
    on="state",
    how="left"
)

print(f"✓ Join complete: {df_enriched.count():,} rows")
print("\nEnriched data schema:")
df_enriched.printSchema()

# Check for nulls in government data columns
null_check = df_enriched.select(
    F.sum(F.when(F.col("total_pos_devices").isNull(), 1).otherwise(0)).alias("null_pos_devices"),
    F.sum(F.when(F.col("total_gov_amount_lakh").isNull(), 1).otherwise(0)).alias("null_gov_amount")
).collect()[0]

print(f"\nNull check after join:")
print(f"  - Rows with null pos_devices: {null_check['null_pos_devices']:,}")
print(f"  - Rows with null gov_amount: {null_check['null_gov_amount']:,}")

Joining transaction log with government infrastructure data...
✓ Join complete: 6,362,620 rows

Enriched data schema:
root
 |-- state: string (nullable = true)
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- is_fraud: long (nullable = true)
 |-- is_flagged_fraud: long (nullable = true)
 |-- total_pos_devices: long (nullable = true)
 |-- total_gov_amount_lakh: double (nullable = true)


Null check after join:
  - Rows with null pos_devices: 0
  - Rows with null gov_amount: 0
